# 📊 Notebook 06: Exportação para Power BI

**Projeto:** XAI-AHP-Gaussian ESGE Framework  
**Autor:** Cesar Yoshio Machado Pedroza  
**Instituição:** USP/Esalq - MBA Data Science and Analytics  
**Data:** 2026-04-16

---

## 🎯 Objetivo

Preparar **Star Schema** (Kimball, 1996) para dashboard no Power BI:

**Tabela Fato:**
- FATO_Scores (histórico de performance)

**Tabelas Dimensão:**
- DIM_Pesos_AHP (robustez da decisão)
- DIM_Importancia_XAI (explicabilidade)
- DIM_Calendario (temporal)

---

## 📖 Fundamento: Star Schema

**Kimball (1996)**: Modelagem dimensional para BI
- Tabela Fato: métricas (facts)
- Tabelas Dimensão: contexto (dimensions)
- Relacionamentos 1:N
- Otimização para queries OLAP

---

## 📚 Referências

- Kimball, R. (1996). *The Data Warehouse Toolkit*. Wiley.

---

In [1]:
"""Setup."""

import sys
from pathlib import Path
import logging
from datetime import datetime

import pandas as pd
import numpy as np

sys.path.append(str(Path.cwd().parent / "src"))
from config import config

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info("🚀 INICIANDO ETL PARA POWER BI")

2026-04-21 00:36:42 | INFO | __main__ | 🚀 INICIANDO ETL PARA POWER BI


## 1️⃣ Carregamento de Dados

In [2]:
"""Carregar datasets."""

# Dataset master
df_master = pd.read_csv(config.DATA_PROCESSED / "esge_master.csv")

# Pesos AHP
df_ahp = pd.read_csv(config.OUTPUTS_TABLES / "ahp_weights.csv")

# Importância SHAP
try:
    df_shap = pd.read_csv(config.OUTPUTS_TABLES / "shap_importance.csv")
except:
    # Fallback: criar dummy
    df_shap = pd.DataFrame({
        'feature': df_master.columns[1:5],
        'importance': [0.334, 0.272, 0.218, 0.177]
    })
    logger.warning("⚠️ SHAP não encontrado, usando valores dummy")

logger.info("✅ Dados carregados")
logger.info(f"  Master: {df_master.shape}")
logger.info(f"  AHP: {df_ahp.shape}")
logger.info(f"  SHAP: {df_shap.shape}")

2026-04-21 00:36:42 | INFO | __main__ | ✅ Dados carregados
2026-04-21 00:36:42 | INFO | __main__ |   Master: (24, 6)
2026-04-21 00:36:42 | INFO | __main__ |   AHP: (4, 6)
2026-04-21 00:36:42 | INFO | __main__ |   SHAP: (4, 2)


## 2️⃣ FATO_Scores (Tabela Fato)

In [3]:
"""Criar tabela fato."""

# Selecionar colunas relevantes
fato_scores = df_master[[
    'year',
    'esg_disclosure_score',
    'annual_return_%',
    'volume',
    'char_count',
    'close_price'
]].copy()

# Adicionar timestamp de atualização
fato_scores['data_atualizacao'] = datetime.now().strftime('%Y-%m-%d')

# Renomear para português (padrão Power BI)
fato_scores = fato_scores.rename(columns={
    'year': 'Ano',
    'esg_disclosure_score': 'Score_ESG',
    'annual_return_%': 'Retorno_Anual_Pct',
    'volume': 'Volume_Negociacao',
    'char_count': 'Qualidade_Relatorio',
    'close_price': 'Preco_Fechamento',
    'data_atualizacao': 'Data_Atualizacao'
})

print("\n📊 FATO_Scores (Preview):")
print(fato_scores.head())

# Exportar
fato_scores.to_csv(
    config.POWERBI_DIR / "FATO_Scores.csv",
    index=False,
    encoding='utf-8-sig'
)
logger.info("✅ FATO_Scores exportada")


📊 FATO_Scores (Preview):
    Ano  Score_ESG  Retorno_Anual_Pct  Volume_Negociacao  Qualidade_Relatorio  \
0  2001          4                NaN       2.002855e+06               140594   
1  2002          3          -7.306216       1.268029e+06               159222   
2  2003          1          91.745497       1.557587e+06               199363   
3  2004          2          70.105323       2.093896e+06               228055   
4  2005          6          70.922119       1.983431e+06               279802   

   Preco_Fechamento Data_Atualizacao  
0          4.227700       2026-04-21  
1          3.918815       2026-04-21  
2          7.514151       2026-04-21  
3         12.781971       2026-04-21  
4         21.847216       2026-04-21  
2026-04-21 00:36:43 | INFO | __main__ | ✅ FATO_Scores exportada


## 3️⃣ DIM_Pesos_AHP (Dimensão)

In [4]:
"""Criar dimensão de pesos AHP."""

dim_pesos = df_ahp.copy()

# Mapear nomes técnicos para dimensões ESGE
mapping = {
    'esg_disclosure_score': 'Environmental (E)',
    'char_count': 'Social (S)',
    'annual_return_%': 'Economic (Ec)',
    'volume': 'Governance (G)'
}

dim_pesos['Criterion'] = dim_pesos['Criterion'].replace(mapping)

# Renomear colunas
dim_pesos = dim_pesos.rename(columns={
    'Criterion': 'Dimensao_ESGE',
    'Mean': 'Peso_Medio',
    'Std': 'Desvio_Padrao',
    'CV_%': 'Coeficiente_Variacao_Pct',
    'CI_95_Lower': 'IC_95_Inferior',
    'CI_95_Upper': 'IC_95_Superior'
})

print("\n⚖️ DIM_Pesos_AHP (Preview):")
print(dim_pesos)

# Exportar
dim_pesos.to_csv(
    config.POWERBI_DIR / "DIM_Pesos_AHP.csv",
    index=False,
    encoding='utf-8-sig'
)
logger.info("✅ DIM_Pesos_AHP exportada")


⚖️ DIM_Pesos_AHP (Preview):
       Dimensao_ESGE  Peso_Medio  Desvio_Padrao  Coeficiente_Variacao_Pct  \
0  Environmental (E)    0.333425       0.014485                  4.344362   
1      Economic (Ec)    0.271611       0.012953                  4.769007   
2     Governance (G)    0.218254       0.011158                  5.112218   
3         Social (S)    0.176710       0.009599                  5.431980   

   IC_95_Inferior  IC_95_Superior  
0        0.305034        0.361815  
1        0.246223        0.297000  
2        0.196385        0.240123  
3        0.157896        0.195524  
2026-04-21 00:36:43 | INFO | __main__ | ✅ DIM_Pesos_AHP exportada


## 4️⃣ DIM_Importancia_XAI (Dimensão)

In [5]:
"""Criar dimensão de importância XAI."""

dim_xai = df_shap.copy()

# Categorizar features em pilares
def categorize_feature(feature):
    """Mapear feature para pilar ESGE."""
    if 'esg' in feature.lower() or 'env' in feature.lower():
        return 'Environmental (E)'
    elif 'char' in feature.lower() or 'social' in feature.lower():
        return 'Social (S)'
    elif 'return' in feature.lower() or 'price' in feature.lower():
        return 'Economic (Ec)'
    elif 'volume' in feature.lower() or 'gov' in feature.lower():
        return 'Governance (G)'
    else:
        return 'Other'

dim_xai['Pilar_ESGE'] = dim_xai['feature'].apply(categorize_feature)

# Renomear
dim_xai = dim_xai.rename(columns={
    'feature': 'Feature',
    'importance': 'Importancia_SHAP'
})

print("\n🔍 DIM_Importancia_XAI (Preview):")
print(dim_xai)

# Exportar
dim_xai.to_csv(
    config.POWERBI_DIR / "DIM_Importancia_XAI.csv",
    index=False,
    encoding='utf-8-sig'
)
logger.info("✅ DIM_Importancia_XAI exportada")


🔍 DIM_Importancia_XAI (Preview):
                Feature  Importancia_SHAP         Pilar_ESGE
0            char_count          7.752377         Social (S)
1  esg_disclosure_score          7.173419  Environmental (E)
2       annual_return_%          5.039273      Economic (Ec)
3                volume          0.062727     Governance (G)
2026-04-21 00:36:43 | INFO | __main__ | ✅ DIM_Importancia_XAI exportada


## 5️⃣ DIM_Calendario (Dimensão Temporal)

In [6]:
"""Criar dimensão de calendário."""

years = sorted(df_master['year'].unique())

dim_calendario = pd.DataFrame({
    'Ano': years,
    'Decada': [f"{(y//10)*10}s" for y in years],
    'Status': [
        'Histórico' if y < datetime.now().year else 'Atual' 
        for y in years
    ],
    'Anos_Desde_Inicio': [y - min(years) for y in years]
})

print("\n📅 DIM_Calendario (Preview):")
print(dim_calendario.head(10))

# Exportar
dim_calendario.to_csv(
    config.POWERBI_DIR / "DIM_Calendario.csv",
    index=False,
    encoding='utf-8-sig'
)
logger.info("✅ DIM_Calendario exportada")


📅 DIM_Calendario (Preview):
    Ano Decada     Status  Anos_Desde_Inicio
0  2001  2000s  Histórico                  0
1  2002  2000s  Histórico                  1
2  2003  2000s  Histórico                  2
3  2004  2000s  Histórico                  3
4  2005  2000s  Histórico                  4
5  2006  2000s  Histórico                  5
6  2007  2000s  Histórico                  6
7  2008  2000s  Histórico                  7
8  2009  2000s  Histórico                  8
9  2010  2010s  Histórico                  9
2026-04-21 00:36:43 | INFO | __main__ | ✅ DIM_Calendario exportada


## 6️⃣ Instruções de Relacionamento

In [7]:
"""Gerar arquivo de instruções."""

instructions = """
═══════════════════════════════════════════════════════════════════════════════
INSTRUÇÕES PARA IMPORTAÇÃO NO POWER BI
═══════════════════════════════════════════════════════════════════════════════

1. IMPORTAR TABELAS:
   - Arquivo → Obter Dados → Texto/CSV
   - Selecionar: FATO_Scores.csv, DIM_*.csv
   - Encoding: UTF-8
   - Delimitador: Vírgula

2. CRIAR RELACIONAMENTOS (Modelo de Dados):

   a) FATO_Scores[Ano] → DIM_Calendario[Ano]
      Cardinalidade: Many-to-One (N:1)
      Direção: Ambas
   
   b) FATO_Scores (via DAX) → DIM_Pesos_AHP
      Uso: Medidas calculadas (ver seção 3)
   
   c) FATO_Scores (via DAX) → DIM_Importancia_XAI
      Uso: Tooltips e análise de drivers

3. MEDIDAS DAX SUGERIDAS:

   Score_ESGE_Ponderado = 
      SUMX(
          DIM_Pesos_AHP,
          DIM_Pesos_AHP[Peso_Medio] * 
          RELATED(FATO_Scores[Score_ESG])  // Ajustar para cada dimensão
      )
   
   Retorno_Medio_Periodo = 
      AVERAGE(FATO_Scores[Retorno_Anual_Pct])
   
   Score_vs_Benchmark = 
      [Score_ESGE_Ponderado] - [Score_Medio_Historico]

4. VISUALIZAÇÕES RECOMENDADAS:

   Dashboard 1: Executivo
   - KPI Cards: Score ESG, Retorno Anual, Volume
   - Line Chart: Evolução temporal (FATO_Scores[Ano] vs Score_ESG)
   - Waterfall: Decomposição AHP (DIM_Pesos_AHP)
   
   Dashboard 2: Análise XAI
   - Bar Chart: Importância SHAP (DIM_Importancia_XAI)
   - Scatter Plot: Retorno vs ESG Score
   - Matrix: Pilar ESGE vs Importância
   
   Dashboard 3: Temporal
   - Slicer: DIM_Calendario[Decada]
   - Timeline: FATO_Scores[Data_Atualizacao]

5. FORMATAÇÃO:

   - Retorno_Anual_Pct: Formato percentual, 2 casas decimais
   - Volume_Negociacao: Formato número, separador de milhares
   - Peso_Medio: Formato decimal, 4 casas
   - Preco_Fechamento: Formato moeda (CAD)

6. FILTROS RECOMENDADOS:

   - DIM_Calendario[Status]: "Histórico" / "Atual"
   - DIM_Pesos_AHP[Dimensao_ESGE]: Filtrar por pilar
   - FATO_Scores[Ano]: Slicer de anos

═══════════════════════════════════════════════════════════════════════════════
SUPORTE:
  Dúvidas: cesar.pedroza@usp.br
  Repositório: github.com/cesarpedroza/teck-esge-xai
═══════════════════════════════════════════════════════════════════════════════
"""

# Salvar instruções
instructions_path = config.POWERBI_DIR / "LEIA-ME_Relacionamentos.txt"
instructions_path.write_text(instructions, encoding='utf-8')

logger.info(f"📄 Instruções salvas: {instructions_path}")

print(instructions)

2026-04-21 00:36:43 | INFO | __main__ | 📄 Instruções salvas: C:\Users\user\Documents\_MBA_Data_Science_Analytics\00 - Temas TCC\teck-esge-xai\powerbi_data\LEIA-ME_Relacionamentos.txt

═══════════════════════════════════════════════════════════════════════════════
INSTRUÇÕES PARA IMPORTAÇÃO NO POWER BI
═══════════════════════════════════════════════════════════════════════════════

1. IMPORTAR TABELAS:
   - Arquivo → Obter Dados → Texto/CSV
   - Selecionar: FATO_Scores.csv, DIM_*.csv
   - Encoding: UTF-8
   - Delimitador: Vírgula

2. CRIAR RELACIONAMENTOS (Modelo de Dados):

   a) FATO_Scores[Ano] → DIM_Calendario[Ano]
      Cardinalidade: Many-to-One (N:1)
      Direção: Ambas

   b) FATO_Scores (via DAX) → DIM_Pesos_AHP
      Uso: Medidas calculadas (ver seção 3)

   c) FATO_Scores (via DAX) → DIM_Importancia_XAI
      Uso: Tooltips e análise de drivers

3. MEDIDAS DAX SUGERIDAS:

   Score_ESGE_Ponderado = 
      SUMX(
          DIM_Pesos_AHP,
          DIM_Pesos_AHP[Peso_Medio] * 
  

## 7️⃣ Sumário de Exportação

In [8]:
"""Sumário final."""

print("\n" + "="*70)
print("✅ EXPORTAÇÃO CONCLUÍDA COM SUCESSO")
print("="*70)
print(f"\n📁 Arquivos criados em: {config.POWERBI_DIR}")
print("\nTabelas:")
print(f"  ✓ FATO_Scores.csv ({len(fato_scores)} linhas)")
print(f"  ✓ DIM_Pesos_AHP.csv ({len(dim_pesos)} linhas)")
print(f"  ✓ DIM_Importancia_XAI.csv ({len(dim_xai)} linhas)")
print(f"  ✓ DIM_Calendario.csv ({len(dim_calendario)} linhas)")
print(f"  ✓ LEIA-ME_Relacionamentos.txt")
print("\n" + "="*70)
print("🚀 Próximo passo: Importar no Power BI")
print("📖 Consulte: LEIA-ME_Relacionamentos.txt")
print("="*70)


✅ EXPORTAÇÃO CONCLUÍDA COM SUCESSO

📁 Arquivos criados em: C:\Users\user\Documents\_MBA_Data_Science_Analytics\00 - Temas TCC\teck-esge-xai\powerbi_data

Tabelas:
  ✓ FATO_Scores.csv (24 linhas)
  ✓ DIM_Pesos_AHP.csv (4 linhas)
  ✓ DIM_Importancia_XAI.csv (4 linhas)
  ✓ DIM_Calendario.csv (24 linhas)
  ✓ LEIA-ME_Relacionamentos.txt

🚀 Próximo passo: Importar no Power BI
📖 Consulte: LEIA-ME_Relacionamentos.txt


## ✅ Checklist

- [ ] FATO_Scores exportada
- [ ] DIM_Pesos_AHP exportada
- [ ] DIM_Importancia_XAI exportada
- [ ] DIM_Calendario exportada
- [ ] Instruções de relacionamento criadas
- [ ] Encoding UTF-8 com BOM
- [ ] Arquivos em powerbi_data/

---

## 📚 Referências

- Kimball, R., & Ross, M. (1996). *The Data Warehouse Toolkit: Practical Techniques for Building Dimensional Data Warehouses*. John Wiley & Sons.

---

**Última Atualização:** 2026-04-16  
**Versão:** 1.0  
**Licença:** MIT